1. Import Libraries & Configuration
2. Load Raw and Phase-1 Processed Data
3. Standardize User Identifiers
4. Prepare Binary Labels
5. Remove Leakage / Invalid Features
6. Define Feature Groups
7. Clean Numerical Features
8. Clean Categorical / Boolean Features
9. Prepare Profile Text
10. Prepare User-level Tweet Data
11. Prepare Temporal / Behavioral Features
12. Prepare Graph Availability Features
13. Build Modality Masks
14. Build Final Labeled Dataset
15. Clean Unlabeled Pseudo-label Pool
16. Stratified Train / Validation / Test Split
17. Fit Preprocessing Only on Train
18. Transform Validation / Test / Unlabeled
19. Export Model-ready Files
20. Final Validation Report

In [3]:
# ============================================================
# 02_data_preprocessing_and_feature_engineering.ipynb
# Cell 1 — Imports & Global Configuration
# ============================================================

from pathlib import Path
import os
import random
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ------------------------------------------------------------
# Project Paths
# Notebook is expected to be inside:
# Bot Detection Implementation/analysis/
# ------------------------------------------------------------

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "processed_data"
RESULTS_DIR = PROJECT_ROOT / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Raw Dataset Paths
# ------------------------------------------------------------

LABELED_USERS_PATH = DATA_DIR / "1000user_sheet - 1k_users.csv"

# Master user pool — contains most of the labeled 1K users too
ALL_USERS_19K_PATH = DATA_DIR / "all_users_new - 19k.csv"

# Follower / Following information collected from the 19K universe
FOLLOWERS_GRAPH_PATH = DATA_DIR / "all_users_new - follower_following.csv"

# Tweet data mainly associated with the labeled 1K users
TWEETS_PATH = DATA_DIR / "1000user_sheet - tweets_meta_data.csv"


# ------------------------------------------------------------
# Phase-1 Processed Files
# ------------------------------------------------------------

LABELED_MODALITY_PATH = (
    PROCESSED_DIR / "labeled_modality_audit.csv"
)

UNLABELED_MODALITY_PATH = (
    PROCESSED_DIR / "unlabeled_modality_audit.csv"
)

TWEET_STATS_PATH = (
    PROCESSED_DIR / "tweet_user_statistics.csv"
)

TWEET_COUNT_PATH = (
    PROCESSED_DIR / "tweet_count_per_user.csv"
)

GRAPH_STATS_PATH = (
    PROCESSED_DIR / "graph_user_statistics.csv"
)

GRAPH_EDGES_PATH = (
    PROCESSED_DIR / "graph_edges.csv"
)

PSEUDO_POOL_PATH = (
    PROCESSED_DIR / "unlabeled_pseudo_label_pool.csv"
)


print("Project root:")
print(PROJECT_ROOT)

print("\nData directory:")
print(DATA_DIR)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print("\nRandom seed:")
print(SEED)

Project root:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation

Data directory:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\data

Processed directory:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\processed_data

Random seed:
42


In [4]:
# ============================================================
# Cell 2 — Load Raw Datasets & Verify Data Sources
# ============================================================

# ------------------------------------------------------------
# Load raw datasets
# ------------------------------------------------------------

labeled_users = pd.read_csv(
    LABELED_USERS_PATH,
    low_memory=False
)

all_users_19k = pd.read_csv(
    ALL_USERS_19K_PATH,
    low_memory=False
)

followers_graph = pd.read_csv(
    FOLLOWERS_GRAPH_PATH,
    low_memory=False
)

tweets = pd.read_csv(
    TWEETS_PATH,
    low_memory=False
)


# ------------------------------------------------------------
# Basic shapes
# ------------------------------------------------------------

print("=" * 70)
print("RAW DATASET SHAPES")
print("=" * 70)

print(f"Labeled users (1K):       {labeled_users.shape}")
print(f"All users (19K):          {all_users_19k.shape}")
print(f"Follower/Following data:  {followers_graph.shape}")
print(f"Tweets:                   {tweets.shape}")


# ------------------------------------------------------------
# Required identifier checks
# ------------------------------------------------------------

datasets = {
    "labeled_users": labeled_users,
    "all_users_19k": all_users_19k,
    "followers_graph": followers_graph,
    "tweets": tweets
}

print("\n" + "=" * 70)
print("IDENTIFIER COLUMNS")
print("=" * 70)

for name, df in datasets.items():
    
    has_id = "id" in df.columns
    has_screen_name = "screen_name" in df.columns
    
    print(
        f"{name:20s} | "
        f"id: {has_id!s:5s} | "
        f"screen_name: {has_screen_name}"
    )


# ------------------------------------------------------------
# Label column detection
# ------------------------------------------------------------

possible_label_columns = [
    "برچسب نهایی",
    "label",
    "class"
]

label_column = None

for col in possible_label_columns:
    if col in labeled_users.columns:
        label_column = col
        break

print("\n" + "=" * 70)
print("LABEL INFORMATION")
print("=" * 70)

print("Detected label column:", label_column)

if label_column is not None:
    print("\nLabel distribution:")
    print(
        labeled_users[label_column]
        .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# Important reminder about dataset roles
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET ROLE CHECK")
print("=" * 70)

print("""
1K Users:
    Ground-truth labeled users.

19K Users:
    Master user pool.
    NOT a purely unlabeled dataset.

Tweets:
    Tweet modality collected mainly for the labeled users.

Follower/Following:
    Network information collected from the wider 19K user universe.

Next step:
    Normalize user identifiers and measure exact overlaps between
    labeled users, tweet users, graph users, and the 19K pool.
""")

RAW DATASET SHAPES
Labeled users (1K):       (1103, 101)
All users (19K):          (19510, 75)
Follower/Following data:  (3453, 4)
Tweets:                   (75913, 38)

IDENTIFIER COLUMNS
labeled_users        | id: True  | screen_name: True
all_users_19k        | id: True  | screen_name: True
followers_graph      | id: True  | screen_name: True
tweets               | id: True  | screen_name: True

LABEL INFORMATION
Detected label column: برچسب نهایی

Label distribution:
برچسب نهایی
human(2)         772
bot(1)           187
can’t find        58
unverified(4)     57
News Agent(3)     29
Name: count, dtype: int64

DATASET ROLE CHECK

1K Users:
    Ground-truth labeled users.

19K Users:
    Master user pool.
    NOT a purely unlabeled dataset.

Tweets:
    Tweet modality collected mainly for the labeled users.

Follower/Following:
    Network information collected from the wider 19K user universe.

Next step:
    Normalize user identifiers and measure exact overlaps between
    labeled u

In [5]:
# ============================================================
# Cell 3 — Standardize User Identifiers & Exact Modality Overlap
# ============================================================

# ------------------------------------------------------------
# 1. Create a canonical user key from screen_name
# ------------------------------------------------------------

def normalize_screen_name(series):
    """
    Normalize Twitter/X screen names for cross-dataset matching.
    - convert to pandas string dtype
    - strip spaces
    - remove leading @
    - lowercase
    - convert empty strings to missing values
    """
    s = series.astype("string")
    s = s.str.strip()
    s = s.str.replace(r"^@", "", regex=True)
    s = s.str.lower()
    s = s.replace("", pd.NA)
    return s


# Do not overwrite original screen_name columns.
labeled_users["user_key"] = normalize_screen_name(
    labeled_users["screen_name"]
)

all_users_19k["user_key"] = normalize_screen_name(
    all_users_19k["screen_name"]
)

followers_graph["user_key"] = normalize_screen_name(
    followers_graph["screen_name"]
)

tweets["user_key"] = normalize_screen_name(
    tweets["screen_name"]
)


# ------------------------------------------------------------
# 2. Basic identifier quality check
# ------------------------------------------------------------

print("=" * 75)
print("IDENTIFIER QUALITY CHECK")
print("=" * 75)

for name, df in datasets.items():
    # datasets was created in Cell 2, but those DataFrames are the same objects
    missing_keys = df["user_key"].isna().sum()
    unique_keys = df["user_key"].nunique(dropna=True)

    print(
        f"{name:20s} | "
        f"rows = {len(df):6d} | "
        f"unique users = {unique_keys:6d} | "
        f"missing user_key = {missing_keys:4d}"
    )


# ------------------------------------------------------------
# 3. Define users represented in each source
# ------------------------------------------------------------

labeled_set = set(
    labeled_users["user_key"].dropna().unique()
)

users_19k_set = set(
    all_users_19k["user_key"].dropna().unique()
)

tweet_user_set = set(
    tweets["user_key"].dropna().unique()
)

graph_record_user_set = set(
    followers_graph["user_key"].dropna().unique()
)


# ------------------------------------------------------------
# 4. Detect graph rows with usable follower/following information
# ------------------------------------------------------------

def has_graph_information(value):
    """
    True if follower/following cell contains usable information.

    Empty / missing representations are treated as unavailable.
    """
    if pd.isna(value):
        return False

    value = str(value).strip()

    invalid_values = {
        "",
        "[]",
        "{}",
        "nan",
        "none",
        "null",
        "<na>"
    }

    return value.lower() not in invalid_values


followers_graph["has_followers_info"] = (
    followers_graph["followers"]
    .apply(has_graph_information)
)

followers_graph["has_following_info"] = (
    followers_graph["following"]
    .apply(has_graph_information)
)

followers_graph["has_usable_graph"] = (
    followers_graph["has_followers_info"]
    |
    followers_graph["has_following_info"]
)


graph_usable_user_set = set(
    followers_graph.loc[
        followers_graph["has_usable_graph"],
        "user_key"
    ]
    .dropna()
    .unique()
)


# ------------------------------------------------------------
# 5. Exact overlap calculations
# ------------------------------------------------------------

labeled_in_19k = labeled_set & users_19k_set

labeled_with_tweets = (
    labeled_set
    & tweet_user_set
)

labeled_with_graph_record = (
    labeled_set
    & graph_record_user_set
)

labeled_with_usable_graph = (
    labeled_set
    & graph_usable_user_set
)

labeled_with_tweets_and_graph = (
    labeled_set
    & tweet_user_set
    & graph_usable_user_set
)

labeled_with_tweets_graph_and_19k = (
    labeled_set
    & tweet_user_set
    & graph_usable_user_set
    & users_19k_set
)


# ------------------------------------------------------------
# 6. Print overlap report
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("EXACT USER OVERLAP")
print("=" * 75)

print(f"Labeled users:                         {len(labeled_set)}")
print(f"19K unique users:                      {len(users_19k_set)}")
print(f"Users with tweets:                     {len(tweet_user_set)}")
print(f"Users with graph record:               {len(graph_record_user_set)}")
print(f"Users with usable graph information:   {len(graph_usable_user_set)}")

print("\n--- Labeled user coverage ---")

print(
    f"Labeled also present in 19K:           "
    f"{len(labeled_in_19k)}"
)

print(
    f"Labeled + Tweets:                      "
    f"{len(labeled_with_tweets)}"
)

print(
    f"Labeled + Graph record:                "
    f"{len(labeled_with_graph_record)}"
)

print(
    f"Labeled + Usable Graph:                "
    f"{len(labeled_with_usable_graph)}"
)

print(
    f"Labeled + Tweets + Usable Graph:       "
    f"{len(labeled_with_tweets_and_graph)}"
)

print(
    f"Labeled + Tweets + Graph + in 19K:     "
    f"{len(labeled_with_tweets_graph_and_19k)}"
)


# ------------------------------------------------------------
# 7. Build temporary complete-multimodal subset
# ------------------------------------------------------------

complete_multimodal_users = labeled_users[
    labeled_users["user_key"].isin(
        labeled_with_tweets_and_graph
    )
].copy()


print("\n" + "=" * 75)
print("LABEL DISTRIBUTION — LABELED + TWEETS + USABLE GRAPH")
print("=" * 75)

print(
    complete_multimodal_users[label_column]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 8. Binary Bot/Human subset inside complete multimodal users
# ------------------------------------------------------------

binary_labels = [
    "human(2)",
    "bot(1)"
]

complete_multimodal_binary = (
    complete_multimodal_users[
        complete_multimodal_users[label_column]
        .isin(binary_labels)
    ]
    .copy()
)


print("\n" + "=" * 75)
print("BOT/HUMAN COMPLETE-MULTIMODAL SUBSET")
print("=" * 75)

print(
    complete_multimodal_binary[label_column]
    .value_counts()
)

print(
    "\nTotal Bot/Human users with "
    "Label + Tweets + Usable Graph:",
    len(complete_multimodal_binary)
)


# ------------------------------------------------------------
# 9. Sanity check
# ------------------------------------------------------------

assert set(
    complete_multimodal_users["user_key"]
).issubset(labeled_set)

assert set(
    complete_multimodal_users["user_key"]
).issubset(tweet_user_set)

assert set(
    complete_multimodal_users["user_key"]
).issubset(graph_usable_user_set)

print("\nSanity checks passed.")

IDENTIFIER QUALITY CHECK
labeled_users        | rows =   1103 | unique users =   1103 | missing user_key =    0
all_users_19k        | rows =  19510 | unique users =  19435 | missing user_key =   75
followers_graph      | rows =   3453 | unique users =   3359 | missing user_key =   10
tweets               | rows =  75913 | unique users =   1099 | missing user_key =    0

EXACT USER OVERLAP
Labeled users:                         1103
19K unique users:                      19435
Users with tweets:                     1099
Users with graph record:               3359
Users with usable graph information:   3359

--- Labeled user coverage ---
Labeled also present in 19K:           1103
Labeled + Tweets:                      1099
Labeled + Graph record:                325
Labeled + Usable Graph:                325
Labeled + Tweets + Usable Graph:       324
Labeled + Tweets + Graph + in 19K:     324

LABEL DISTRIBUTION — LABELED + TWEETS + USABLE GRAPH
برچسب نهایی
human(2)         232
bot(1)  

In [6]:
# ============================================================
# Cell 4 — Verify Labeled 1K ↔ 19K Consistency
# ============================================================

# ------------------------------------------------------------
# 1. Check duplicate behavior before joining
# ------------------------------------------------------------

print("=" * 75)
print("DUPLICATE USER_KEY CHECK")
print("=" * 75)

for name, df in {
    "labeled_users": labeled_users,
    "all_users_19k": all_users_19k,
    "followers_graph": followers_graph,
}.items():

    valid = df[df["user_key"].notna()].copy()

    duplicate_rows = valid.duplicated(
        subset="user_key",
        keep=False
    ).sum()

    duplicate_users = (
        valid.loc[
            valid.duplicated("user_key", keep=False),
            "user_key"
        ]
        .nunique()
    )

    print(
        f"{name:20s} | "
        f"duplicate rows = {duplicate_rows:4d} | "
        f"duplicate users = {duplicate_users:4d}"
    )


# ------------------------------------------------------------
# 2. Build exact labeled / true-unlabeled partition
# ------------------------------------------------------------

valid_19k = all_users_19k[
    all_users_19k["user_key"].notna()
].copy()

true_unlabeled_19k = valid_19k[
    ~valid_19k["user_key"].isin(labeled_set)
].copy()


print("\n" + "=" * 75)
print("19K PARTITION")
print("=" * 75)

print(
    f"Valid unique users in 19K:    "
    f"{valid_19k['user_key'].nunique()}"
)

print(
    f"Labeled users inside 19K:     "
    f"{len(labeled_in_19k)}"
)

print(
    f"True unlabeled unique users:  "
    f"{true_unlabeled_19k['user_key'].nunique()}"
)


# ------------------------------------------------------------
# 3. Ensure zero overlap
# ------------------------------------------------------------

true_unlabeled_set = set(
    true_unlabeled_19k["user_key"].unique()
)

intersection = labeled_set & true_unlabeled_set

print(
    f"Labeled ∩ True Unlabeled:     "
    f"{len(intersection)}"
)

assert len(intersection) == 0


# ------------------------------------------------------------
# 4. Find columns shared between labeled 1K and 19K
# ------------------------------------------------------------

excluded_columns = {
    "user_key",
    label_column
}

shared_columns = sorted(
    (
        set(labeled_users.columns)
        & set(all_users_19k.columns)
    )
    - excluded_columns
)

print("\n" + "=" * 75)
print("SHARED COLUMNS BETWEEN 1K AND 19K")
print("=" * 75)

print(f"Number of shared columns: {len(shared_columns)}")

print("\nShared columns:")
print(shared_columns)


# ------------------------------------------------------------
# 5. Merge the 1103 overlapping users for consistency analysis
# ------------------------------------------------------------

comparison = labeled_users[
    ["user_key"] + shared_columns
].merge(
    valid_19k[
        ["user_key"] + shared_columns
    ],
    on="user_key",
    how="inner",
    suffixes=("_1k", "_19k"),
    validate="one_to_one"
)

print("\nMatched users:", len(comparison))


# ------------------------------------------------------------
# 6. Compare missingness and exact agreement per shared feature
# ------------------------------------------------------------

comparison_report = []

for col in shared_columns:

    col_1k = f"{col}_1k"
    col_19k = f"{col}_19k"

    s1 = comparison[col_1k]
    s2 = comparison[col_19k]

    both_present = s1.notna() & s2.notna()

    comparable_count = int(
        both_present.sum()
    )

    # Convert to normalized strings only for equality comparison.
    # We are not modifying the original values.
    a = (
        s1[both_present]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    b = (
        s2[both_present]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    agreement_count = int(
        (a == b).sum()
    )

    if comparable_count > 0:
        agreement_rate = (
            agreement_count
            / comparable_count
            * 100
        )
    else:
        agreement_rate = np.nan

    comparison_report.append({
        "feature": col,

        "missing_1k": int(
            s1.isna().sum()
        ),

        "missing_19k": int(
            s2.isna().sum()
        ),

        "both_present": comparable_count,

        "exact_agreement_count":
            agreement_count,

        "exact_agreement_pct":
            agreement_rate
    })


comparison_report = pd.DataFrame(
    comparison_report
).sort_values(
    by="exact_agreement_pct",
    ascending=True,
    na_position="last"
)


# ------------------------------------------------------------
# 7. Display features with lowest agreement first
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("1K ↔ 19K FEATURE CONSISTENCY")
print("=" * 75)

display(
    comparison_report.head(30)
)


print("\n" + "=" * 75)
print("CHECK PASSED")
print("=" * 75)

print(
    "The labeled and true-unlabeled populations "
    "are completely separated."
)

DUPLICATE USER_KEY CHECK
labeled_users        | duplicate rows =    0 | duplicate users =    0
all_users_19k        | duplicate rows =    0 | duplicate users =    0
followers_graph      | duplicate rows =  161 | duplicate users =   77

19K PARTITION
Valid unique users in 19K:    19435
Labeled users inside 19K:     1103
True unlabeled unique users:  18332
Labeled ∩ True Unlabeled:     0

SHARED COLUMNS BETWEEN 1K AND 19K
Number of shared columns: 68

Shared columns:
['Botometer Score', 'Grok 4', 'LLM(ChatGPT)', 'LLM(Gemini)', 'clean_description', 'cluster_id', 'created_at', 'default_profile', 'default_profile_image', 'description', 'fast_followers_count', 'favourites_count', 'follower_growth_rate', 'followers_count', 'following', 'friends_count', 'friends_growth_rate', 'has_custom_timelines', 'hashtag_in_description', 'id', 'is_translator', 'listed_count', 'location', 'max_occurence_of_same_gap', 'max_tweets_per_day', 'max_tweets_per_hour', 'mean_favourites_per_tweet', 'mean_no_hashtags

,feature,missing_1k,missing_19k,both_present,exact_agreement_count,exact_agreement_pct
7,default_profile,0,0,1103,0,0.000000
14,following,395,395,708,0,0.000000
11,favourites_count,0,0,1103,0,0.000000
15,friends_count,0,2,1101,0,0.000000
13,followers_count,0,2,1101,0,0.000000
10,fast_followers_count,0,0,1103,0,0.000000
32,media_count,0,0,1103,0,0.000000
21,listed_count,0,0,1103,0,0.000000
51,statuses_count,0,0,1103,0,0.000000
58,verified,0,0,1103,0,0.000000



CHECK PASSED
The labeled and true-unlabeled populations are completely separated.


In [7]:
# ============================================================
# Cell 5 — Type-Aware 1K ↔ 19K Feature Consistency
# ============================================================

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def normalize_boolean_value(x):
    """
    Normalize common boolean representations to 0 / 1.
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    true_values = {
        "true", "1", "1.0", "yes", "y"
    }

    false_values = {
        "false", "0", "0.0", "no", "n"
    }

    if s in true_values:
        return 1

    if s in false_values:
        return 0

    return np.nan


def normalized_string_series(series):
    """
    Normalize text only for comparison.
    Does not modify original data.
    """
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
    )


def numeric_conversion_rate(series):
    """
    Fraction of non-missing values that can be converted to numeric.
    """
    non_missing = series.dropna()

    if len(non_missing) == 0:
        return 0.0

    numeric = pd.to_numeric(
        non_missing,
        errors="coerce"
    )

    return numeric.notna().mean()


def boolean_conversion_rate(series):
    """
    Fraction of non-missing values that look boolean.
    """
    non_missing = series.dropna()

    if len(non_missing) == 0:
        return 0.0

    converted = non_missing.apply(
        normalize_boolean_value
    )

    return converted.notna().mean()


# ------------------------------------------------------------
# Type-aware comparison
# ------------------------------------------------------------

semantic_report = []

for col in shared_columns:

    s1 = comparison[f"{col}_1k"]
    s2 = comparison[f"{col}_19k"]

    both_present = (
        s1.notna()
        & s2.notna()
    )

    n = int(both_present.sum())

    if n == 0:
        semantic_report.append({
            "feature": col,
            "comparison_type": "no comparable data",
            "both_present": 0,
            "agreement_count": 0,
            "agreement_pct": np.nan,
            "missing_1k": int(s1.isna().sum()),
            "missing_19k": int(s2.isna().sum())
        })

        continue

    a = s1[both_present]
    b = s2[both_present]

    # --------------------------------------------------------
    # Detect whether feature behaves like boolean
    # --------------------------------------------------------

    bool_rate_a = boolean_conversion_rate(a)
    bool_rate_b = boolean_conversion_rate(b)

    num_rate_a = numeric_conversion_rate(a)
    num_rate_b = numeric_conversion_rate(b)

    if (
        bool_rate_a >= 0.95
        and bool_rate_b >= 0.95
    ):

        comparison_type = "boolean"

        aa = a.apply(
            normalize_boolean_value
        )

        bb = b.apply(
            normalize_boolean_value
        )

        matches = (
            aa.values == bb.values
        )

    # --------------------------------------------------------
    # Numeric comparison
    # --------------------------------------------------------

    elif (
        num_rate_a >= 0.95
        and num_rate_b >= 0.95
    ):

        comparison_type = "numeric"

        aa = pd.to_numeric(
            a,
            errors="coerce"
        )

        bb = pd.to_numeric(
            b,
            errors="coerce"
        )

        matches = np.isclose(
            aa.values,
            bb.values,
            rtol=1e-6,
            atol=1e-8,
            equal_nan=True
        )

    # --------------------------------------------------------
    # Text comparison
    # --------------------------------------------------------

    else:

        comparison_type = "text"

        aa = normalized_string_series(a)
        bb = normalized_string_series(b)

        matches = (
            aa.values == bb.values
        )

    agreement_count = int(
        np.sum(matches)
    )

    agreement_pct = (
        agreement_count
        / n
        * 100
    )

    semantic_report.append({
        "feature": col,
        "comparison_type": comparison_type,
        "both_present": n,
        "agreement_count": agreement_count,
        "agreement_pct": agreement_pct,
        "missing_1k": int(s1.isna().sum()),
        "missing_19k": int(s2.isna().sum())
    })


semantic_report = pd.DataFrame(
    semantic_report
).sort_values(
    by="agreement_pct",
    ascending=True,
    na_position="last"
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 75)
print("TYPE-AWARE 1K ↔ 19K CONSISTENCY")
print("=" * 75)

display(
    semantic_report.head(30)
)


# ------------------------------------------------------------
# High-agreement features
# ------------------------------------------------------------

high_agreement = semantic_report[
    semantic_report["agreement_pct"] >= 99
]

print("\n" + "=" * 75)
print("FEATURES WITH >= 99% AGREEMENT")
print("=" * 75)

print(
    f"{len(high_agreement)} / "
    f"{len(semantic_report)} shared features"
)


# ------------------------------------------------------------
# Low-agreement features
# ------------------------------------------------------------

low_agreement = semantic_report[
    semantic_report["agreement_pct"] < 95
]

print("\n" + "=" * 75)
print("FEATURES WITH < 95% AGREEMENT")
print("=" * 75)

display(
    low_agreement[
        [
            "feature",
            "comparison_type",
            "both_present",
            "agreement_pct",
            "missing_1k",
            "missing_19k"
        ]
    ]
)

TYPE-AWARE 1K ↔ 19K CONSISTENCY


,feature,comparison_type,both_present,agreement_count,agreement_pct,missing_1k,missing_19k
67,آیا پروفایل واقعی به‌نظر می‌رسد؟عکس واقعی، بیو...,boolean,68,67,98.529412,1035,1015
61,آیا تعامل با دیگران وجود دارد؟ریپلای، منشن، گف...,boolean,69,68,98.550725,1034,1014
66,آیا محتوای توییت‌ها خودکار به‌نظر می‌رسد؟تکرار...,boolean,70,69,98.571429,1033,1013
64,آیا رفتار زمانی توییت‌ها طبیعی است؟ارسال نامنظ...,boolean,70,69,98.571429,1033,1013
65,آیا فرکانس استفاده از یک هشتگ زیاد است,boolean,129,128,99.224806,974,935
22,location,text,639,635,99.374022,464,464
62,آیا توییت‌ها متنوع و طبیعی‌اند؟زبان انسانی، نظ...,boolean,172,171,99.418605,931,876
35,name,text,1103,1101,99.818676,0,0
15,friends_count,numeric,1101,1100,99.909173,0,2
13,followers_count,numeric,1101,1100,99.909173,0,2



FEATURES WITH >= 99% AGREEMENT
63 / 68 shared features

FEATURES WITH < 95% AGREEMENT


,feature,comparison_type,both_present,agreement_pct,missing_1k,missing_19k


In [8]:
# ============================================================
# Cell 6 — Column Provenance & Leakage Audit
# ============================================================

# ------------------------------------------------------------
# 1. Determine column provenance
# ------------------------------------------------------------

base_exclusions = {"user_key"}

columns_1k = set(labeled_users.columns) - base_exclusions
columns_19k = set(all_users_19k.columns) - base_exclusions

only_1k_columns = sorted(
    columns_1k - columns_19k
)

only_19k_columns = sorted(
    columns_19k - columns_1k
)

shared_columns_full = sorted(
    columns_1k & columns_19k
)


print("=" * 80)
print("COLUMN PROVENANCE")
print("=" * 80)

print(f"Columns only in labeled 1K: {len(only_1k_columns)}")
print(f"Columns only in 19K:        {len(only_19k_columns)}")
print(f"Shared columns:             {len(shared_columns_full)}")


# ------------------------------------------------------------
# 2. Display columns existing only in labeled 1K
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COLUMNS ONLY IN LABELED 1K")
print("=" * 80)

only_1k_report = []

for col in only_1k_columns:

    only_1k_report.append({
        "column": col,
        "dtype": str(labeled_users[col].dtype),
        "missing_count": int(
            labeled_users[col].isna().sum()
        ),
        "missing_pct": round(
            labeled_users[col].isna().mean() * 100,
            2
        ),
        "n_unique": int(
            labeled_users[col].nunique(
                dropna=True
            )
        )
    })


only_1k_report = pd.DataFrame(
    only_1k_report
)

display(only_1k_report)


# ------------------------------------------------------------
# 3. Display columns existing only in 19K
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COLUMNS ONLY IN 19K")
print("=" * 80)

only_19k_report = []

for col in only_19k_columns:

    only_19k_report.append({
        "column": col,
        "dtype": str(all_users_19k[col].dtype),
        "missing_count": int(
            all_users_19k[col].isna().sum()
        ),
        "missing_pct": round(
            all_users_19k[col].isna().mean() * 100,
            2
        ),
        "n_unique": int(
            all_users_19k[col].nunique(
                dropna=True
            )
        )
    })


only_19k_report = pd.DataFrame(
    only_19k_report
)

display(only_19k_report)


# ------------------------------------------------------------
# 4. Automatically flag suspicious / leakage-prone columns
# ------------------------------------------------------------

all_candidate_columns = sorted(
    set(labeled_users.columns)
    | set(all_users_19k.columns)
)

leakage_keywords = [
    "label",
    "class",
    "llm",
    "chatgpt",
    "gemini",
    "grok",
    "tagger",
    "prob",
    "reason",
    "botometer",
    "برچسب"
]

manual_review_prefixes = [
    "آیا "
]


def is_suspicious_column(col):

    col_lower = str(col).lower()

    keyword_match = any(
        keyword in col_lower
        for keyword in leakage_keywords
    )

    manual_match = any(
        str(col).startswith(prefix)
        for prefix in manual_review_prefixes
    )

    unnamed_match = str(col).lower().startswith(
        "unnamed"
    )

    return (
        keyword_match
        or manual_match
        or unnamed_match
    )


suspected_leakage_columns = [
    col
    for col in all_candidate_columns
    if is_suspicious_column(col)
]


print("\n" + "=" * 80)
print("SUSPECTED LEAKAGE / MANUAL-JUDGMENT COLUMNS")
print("=" * 80)

for i, col in enumerate(
    suspected_leakage_columns,
    start=1
):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 5. Define columns that must NEVER be direct model inputs
# ------------------------------------------------------------

hard_exclude_columns = set(
    suspected_leakage_columns
)

hard_exclude_columns.update({
    label_column,
    "id",
    "screen_name",
    "user_key"
})


print("\n" + "=" * 80)
print("HARD-EXCLUDE COLUMN COUNT")
print("=" * 80)

print(
    len(hard_exclude_columns),
    "columns currently marked as non-model inputs."
)


# ------------------------------------------------------------
# 6. Important note
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NOTE")
print("=" * 80)

print("""
This cell does NOT delete any column.

It only identifies:
1) columns unique to the labeled dataset,
2) columns unique to the 19K master pool,
3) potential target leakage / external detector outputs,
4) identifiers that must not be used as predictive features.

Final feature selection will be done explicitly in the next cells.
""")

COLUMN PROVENANCE
Columns only in labeled 1K: 33
Columns only in 19K:        7
Shared columns:             68

COLUMNS ONLY IN LABELED 1K


,column,dtype,missing_count,missing_pct,n_unique
0,Probe2,float64,0,0.00,11
1,Probe3,float64,0,0.00,11
2,Reason2,str,19,1.72,913
3,Reason3,str,28,2.54,733
4,Unnamed: 58,float64,1103,100.00,0
5,"class (1:bot, 2:human, 3:News Agent, 4:unverif...",float64,1,0.09,5
6,"class (1:bot, 2:human, 3:News Agent, 4:unverif...",int64,0,0.00,4
7,"class (1:bot, 2:human, 3:News Agent, 4:unverif...",int64,0,0.00,4
8,description_length,int64,0,0.00,164
9,followers_friend_ratio,float64,0,0.00,1103



COLUMNS ONLY IN 19K


,column,dtype,missing_count,missing_pct,n_unique
0,last_updated,str,76,0.39,19273
1,prob 2,float64,19009,97.43,20
2,prob1,str,17584,90.13,15
3,reason 1,str,17629,90.36,1505
4,reason 2,float64,19510,100.00,0
5,tagger's name 1,str,17466,89.52,9
6,tagger's name2,str,19009,97.43,1



SUSPECTED LEAKAGE / MANUAL-JUDGMENT COLUMNS
01. Botometer Score
02. Grok 4
03. LLM(ChatGPT)
04. LLM(Gemini)
05. Probe2
06. Probe3
07. Reason2
08. Reason3
09. Unnamed: 58
10. class (1:bot, 2:human, 3:News Agent, 4:unverified)
11. class (1:bot, 2:human, 3:News Agent, 4:unverified)_1
12. class (1:bot, 2:human, 3:News Agent, 4:unverified)_2
13. prob 2
14. prob1
15. reason 1
16. reason 2
17. tagger's name 1
18. tagger's name 2
19. tagger's name 3
20. tagger's name2
21. آیا تعامل با دیگران وجود دارد؟ریپلای، منشن، گفت‌وگو با کاربران (ویژگی‌های رفتاری در مکالمه (Conversational Features)
•        پاسخ‌دهی کم: حسابی که تقریباً هیچوقت جواب مستقیم نمی‌دهد.
•        پاسخ‌های ماشینی: جملات کلیشه‌ای و تکراری مثل "Great post!" یا "Nice info"
•        تأخیر یکنواخت در پاسخ: مثلاً همیشه ۳ ثانیه بعد جواب می‌دهد.
•        نبود تعامل شخصی: اشاره نکردن به تجربیات فردی یا مکالمات واقعی.)
22. آیا توییت‌ها متنوع و طبیعی‌اند؟زبان انسانی، نظر شخصی، طنز، تعامل
23. آیا دنبال‌شوندگان/دنبال‌کننده‌ها غیرعادی‌اند؟نسب

In [9]:
# ============================================================
# Cell 7 — Define Canonical Feature Groups
# ============================================================

# ============================================================
# IMPORTANT DESIGN DECISION
# ============================================================
#
# 19K will be the canonical source of user features because:
#   1) all 1103 labeled users exist inside it,
#   2) shared features are highly consistent,
#   3) it has fewer missing values for several features,
#   4) labeled and unlabeled users will therefore use the
#      same feature definitions and data snapshot.
#
# Ground-truth labels will still come ONLY from the 1K dataset.
# ============================================================


# ------------------------------------------------------------
# 1. Identifier / target columns
# ------------------------------------------------------------

IDENTIFIER_COLUMNS = [
    "id",
    "screen_name",
    "user_key"
]

TARGET_COLUMN = label_column


# ------------------------------------------------------------
# 2. External detector / leakage columns
# NEVER use these as model inputs
# ------------------------------------------------------------

EXTERNAL_DETECTOR_COLUMNS = [
    "Botometer Score",
    "LLM(ChatGPT)",
    "LLM(Gemini)",
    "Grok 4"
]


# ------------------------------------------------------------
# 3. Columns from annotation / taggers / manual judgments
# ------------------------------------------------------------

ANNOTATION_KEYWORDS = [
    "tagger",
    "prob",
    "reason",
    "probe",
    "class",
    "برچسب"
]


def is_annotation_column(col):

    c = str(col).strip()
    c_lower = c.lower()

    # LLM/manual questionnaire columns
    if c.startswith("آیا "):
        return True

    # tagger / probability / reason / class columns
    if any(
        keyword in c_lower
        for keyword in ANNOTATION_KEYWORDS
    ):
        return True

    return False


ANNOTATION_COLUMNS = [
    col
    for col in all_users_19k.columns
    if is_annotation_column(col)
]


# ------------------------------------------------------------
# 4. Raw fields retained for reference but NOT used directly
#    as tabular model features
# ------------------------------------------------------------

RAW_REFERENCE_COLUMNS = [
    "created_at",              # user_age is preferable
    "location",                # high-cardinality free text
    "name",                    # derive numeric characteristics instead
    "description",             # used only for text processing
    "clean_description",       # used only for text processing
    "url",
    "profile_banner_url",
    "profile_image_url_https",
    "pinned_tweet_ids_str",
    "cluster_id",              # dataset/community artifact
    "following",               # API account relation, NOT graph adjacency
    "last_updated"
]


# ------------------------------------------------------------
# 5. Very sparse / unsuitable raw profile fields
# ------------------------------------------------------------

SPARSE_OR_UNSTABLE_COLUMNS = [
    "profile_interstitial_type",
    "verified_type"
]


# ------------------------------------------------------------
# 6. Profile numerical features
# ------------------------------------------------------------

PROFILE_NUMERIC_FEATURES = [
    "followers_count",
    "friends_count",
    "favourites_count",
    "listed_count",
    "media_count",
    "statuses_count",
    "status_count",
    "fast_followers_count",
    "normal_followers_count",
    "user_age",
    "follower_growth_rate",
    "friends_growth_rate"
]


# ------------------------------------------------------------
# 7. Profile binary / categorical features
# ------------------------------------------------------------

PROFILE_BINARY_FEATURES = [
    "default_profile",
    "default_profile_image",
    "verified",
    "has_custom_timelines",
    "is_translator",
    "possibly_sensitive",
    "want_retweets",
    "hashtag_in_description",
    "numbers_in_description"
]


PROFILE_CATEGORICAL_FEATURES = [
    "translator_type"
]


# ------------------------------------------------------------
# 8. Behavioral / temporal features
# ------------------------------------------------------------

BEHAVIOR_TEMPORAL_FEATURES = [
    "no_type_tweet",
    "no_type_retweet_with_comment",
    "no_type_reply",

    "mean_no_media_per_tweet",
    "mean_no_words",
    "no_languages",
    "mean_no_hashtags",
    "mean_favourites_per_tweet",

    "time_between_tweets",
    "tweet_frequency",

    "min_tweets_per_hour",
    "min_tweets_per_day",
    "max_tweets_per_hour",
    "max_tweets_per_day",

    "max_occurence_of_same_gap",

    "unique_mention_rate_per_tweet",
    "mean_user_mentions_per_tweet",

    "retweet_as_tweet_rate",
    "no_retweet_tweets",
    "mean_retweets_per_tweet"
]


# ------------------------------------------------------------
# 9. Text modality fields
# ------------------------------------------------------------

TEXT_FEATURES = [
    "clean_description",
    "description"
]

# Tweet text itself remains in the separate tweets dataframe.
# It will NOT be flattened into the master user table.


# ------------------------------------------------------------
# 10. Reproducible engineered features
# These existed only in 1K, but we will recreate them
# ourselves for BOTH labeled and unlabeled users.
# ------------------------------------------------------------

DERIVED_FEATURES_TO_CREATE = [
    "description_length",
    "followers_friend_ratio",
    "listed_growth_rate",
    "num_digits_in_name",
    "num_digits_in_username",
    "url_in_description"
]


# ------------------------------------------------------------
# 11. 1K-only features NOT used directly
# ------------------------------------------------------------

SAFE_REDERIVABLE_1K_COLUMNS = set(
    DERIVED_FEATURES_TO_CREATE
)

ONE_K_ONLY_NOT_DIRECTLY_USED = sorted(
    set(only_1k_columns)
    - SAFE_REDERIVABLE_1K_COLUMNS
)


# ------------------------------------------------------------
# 12. Complete candidate feature collection
# ------------------------------------------------------------

BASE_TABULAR_FEATURES = (
    PROFILE_NUMERIC_FEATURES
    + PROFILE_BINARY_FEATURES
    + PROFILE_CATEGORICAL_FEATURES
    + BEHAVIOR_TEMPORAL_FEATURES
)


# ------------------------------------------------------------
# 13. Validate that expected features exist in 19K
# ------------------------------------------------------------

expected_columns = set(
    BASE_TABULAR_FEATURES
    + TEXT_FEATURES
)

missing_expected_columns = sorted(
    expected_columns
    - set(all_users_19k.columns)
)

print("=" * 80)
print("FEATURE GROUP VALIDATION")
print("=" * 80)

print(
    "Profile numeric features:      ",
    len(PROFILE_NUMERIC_FEATURES)
)

print(
    "Profile binary features:       ",
    len(PROFILE_BINARY_FEATURES)
)

print(
    "Profile categorical features:  ",
    len(PROFILE_CATEGORICAL_FEATURES)
)

print(
    "Behavior/temporal features:    ",
    len(BEHAVIOR_TEMPORAL_FEATURES)
)

print(
    "Base tabular features total:   ",
    len(BASE_TABULAR_FEATURES)
)

print(
    "Derived features to create:    ",
    len(DERIVED_FEATURES_TO_CREATE)
)

print(
    "Text fields:                   ",
    len(TEXT_FEATURES)
)


print("\nMissing expected columns:")

if missing_expected_columns:
    print(missing_expected_columns)
else:
    print("None")


assert len(missing_expected_columns) == 0


# ------------------------------------------------------------
# 14. Check accidental leakage
# ------------------------------------------------------------

selected_model_columns = set(
    BASE_TABULAR_FEATURES
)

forbidden_columns = (
    set(EXTERNAL_DETECTOR_COLUMNS)
    | set(ANNOTATION_COLUMNS)
    | set(IDENTIFIER_COLUMNS)
    | set(SPARSE_OR_UNSTABLE_COLUMNS)
)

leakage_overlap = (
    selected_model_columns
    & forbidden_columns
)

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

print(
    "Forbidden columns accidentally selected:",
    leakage_overlap
)

assert len(leakage_overlap) == 0


# ------------------------------------------------------------
# 15. Missingness summary for candidate tabular features
# ------------------------------------------------------------

feature_missingness = pd.DataFrame({
    "feature": BASE_TABULAR_FEATURES,
    "missing_count": [
        all_users_19k[col].isna().sum()
        for col in BASE_TABULAR_FEATURES
    ],
    "missing_pct": [
        all_users_19k[col].isna().mean() * 100
        for col in BASE_TABULAR_FEATURES
    ],
    "dtype": [
        str(all_users_19k[col].dtype)
        for col in BASE_TABULAR_FEATURES
    ]
}).sort_values(
    "missing_pct",
    ascending=False
)


print("\n" + "=" * 80)
print("CANDIDATE FEATURE MISSINGNESS — 19K MASTER POOL")
print("=" * 80)

display(feature_missingness)


print("\nFeature policy successfully defined.")

FEATURE GROUP VALIDATION
Profile numeric features:       12
Profile binary features:        9
Profile categorical features:   1
Behavior/temporal features:     20
Base tabular features total:    42
Derived features to create:     6
Text fields:                    2

Missing expected columns:
None

LEAKAGE CHECK
Forbidden columns accidentally selected: set()

CANDIDATE FEATURE MISSINGNESS — 19K MASTER POOL


,feature,missing_count,missing_pct,dtype
7,fast_followers_count,76,0.389544,float64
5,statuses_count,76,0.389544,float64
8,normal_followers_count,76,0.389544,float64
15,has_custom_timelines,76,0.389544,object
17,possibly_sensitive,76,0.389544,object
21,translator_type,76,0.389544,str
18,want_retweets,76,0.389544,object
16,is_translator,76,0.389544,object
13,default_profile_image,73,0.374167,object
0,followers_count,5,0.025628,float64



Feature policy successfully defined.


In [10]:
# ============================================================
# Cell 8 — Recreate & Validate Reproducible Derived Features
# ============================================================

# ------------------------------------------------------------
# 1. Start canonical working user table from valid 19K users
# ------------------------------------------------------------

users_master = valid_19k.copy()

print("=" * 80)
print("CANONICAL USER TABLE")
print("=" * 80)

print("Rows:", len(users_master))
print(
    "Unique users:",
    users_master["user_key"].nunique()
)

assert len(users_master) == 19435
assert users_master["user_key"].is_unique


# ------------------------------------------------------------
# 2. Helper — count Unicode digits
# ------------------------------------------------------------

def count_digits(value):

    if pd.isna(value):
        return 0

    return sum(
        char.isdigit()
        for char in str(value)
    )


# ------------------------------------------------------------
# 3. description_length
# Same definition as the existing 1K feature:
# number of characters in raw description
# ------------------------------------------------------------

users_master["description_length"] = (
    users_master["description"]
    .fillna("")
    .astype(str)
    .str.len()
)


# ------------------------------------------------------------
# 4. Number of digits in display name
# Unicode-aware: captures 4, ۴, ⁴, etc.
# ------------------------------------------------------------

users_master["num_digits_in_name"] = (
    users_master["name"]
    .apply(count_digits)
)


# ------------------------------------------------------------
# 5. Number of digits in username
# ------------------------------------------------------------

users_master["num_digits_in_username"] = (
    users_master["screen_name"]
    .apply(count_digits)
)


# ------------------------------------------------------------
# 6. URL in description
# ------------------------------------------------------------

users_master["url_in_description"] = (
    users_master["description"]
    .fillna("")
    .astype(str)
    .str.contains(
        r"https?://|www\.",
        case=False,
        regex=True
    )
    .astype(int)
)


# ------------------------------------------------------------
# 7. Followers / friends ratio
#
# Existing 1K behavior:
# followers_count / friends_count
#
# If friends_count == 0:
# use followers_count instead of infinity.
# ------------------------------------------------------------

followers = pd.to_numeric(
    users_master["followers_count"],
    errors="coerce"
)

friends = pd.to_numeric(
    users_master["friends_count"],
    errors="coerce"
)

users_master["followers_friend_ratio"] = np.where(
    friends > 0,
    followers / friends,
    followers
)


# ------------------------------------------------------------
# 8. Final reproducible derived feature list
# ------------------------------------------------------------

DERIVED_FEATURES = [
    "description_length",
    "followers_friend_ratio",
    "num_digits_in_name",
    "num_digits_in_username",
    "url_in_description"
]


print("\nDerived features created:")

for feature in DERIVED_FEATURES:
    print(" -", feature)


# ------------------------------------------------------------
# 9. Validate against existing values in labeled 1K
# ------------------------------------------------------------

validation_columns = [
    "user_key"
] + DERIVED_FEATURES

derived_validation = labeled_users[
    validation_columns
].merge(
    users_master[
        validation_columns
    ],
    on="user_key",
    how="inner",
    suffixes=("_1k", "_new"),
    validate="one_to_one"
)


validation_results = []

for feature in DERIVED_FEATURES:

    old = derived_validation[
        f"{feature}_1k"
    ]

    new = derived_validation[
        f"{feature}_new"
    ]

    valid = old.notna() & new.notna()

    if feature == "followers_friend_ratio":

        matches = np.isclose(
            pd.to_numeric(
                old[valid],
                errors="coerce"
            ),
            pd.to_numeric(
                new[valid],
                errors="coerce"
            ),
            rtol=1e-5,
            atol=1e-6
        )

    else:

        matches = (
            old[valid].values
            ==
            new[valid].values
        )

    agreement = (
        matches.mean() * 100
        if len(matches) > 0
        else np.nan
    )

    validation_results.append({
        "feature": feature,
        "compared_users": int(valid.sum()),
        "agreement_pct": agreement
    })


derived_validation_report = pd.DataFrame(
    validation_results
)


print("\n" + "=" * 80)
print("DERIVED FEATURE VALIDATION AGAINST 1K")
print("=" * 80)

display(
    derived_validation_report
)


# ------------------------------------------------------------
# 10. Final shared tabular feature set
# ------------------------------------------------------------

FINAL_TABULAR_FEATURES = (
    BASE_TABULAR_FEATURES
    + DERIVED_FEATURES
)


print("\n" + "=" * 80)
print("FINAL SHARED TABULAR FEATURE SPACE")
print("=" * 80)

print(
    "Base features:",
    len(BASE_TABULAR_FEATURES)
)

print(
    "Derived features:",
    len(DERIVED_FEATURES)
)

print(
    "Total tabular features:",
    len(FINAL_TABULAR_FEATURES)
)


# ------------------------------------------------------------
# 11. Safety checks
# ------------------------------------------------------------

assert all(
    feature in users_master.columns
    for feature in FINAL_TABULAR_FEATURES
)

assert (
    users_master["user_key"]
    .nunique()
    ==
    len(users_master)
)

print("\nDerived feature construction completed successfully.")

CANONICAL USER TABLE
Rows: 19435
Unique users: 19435

Derived features created:
 - description_length
 - followers_friend_ratio
 - num_digits_in_name
 - num_digits_in_username
 - url_in_description

DERIVED FEATURE VALIDATION AGAINST 1K


,feature,compared_users,agreement_pct
0,description_length,1103,100.000000
1,followers_friend_ratio,1101,99.909173
2,num_digits_in_name,1103,100.000000
3,num_digits_in_username,1103,100.000000
4,url_in_description,1103,100.000000



FINAL SHARED TABULAR FEATURE SPACE
Base features: 42
Derived features: 5
Total tabular features: 47

Derived feature construction completed successfully.


In [11]:
# ============================================================
# Cell 9 — Build Canonical Labeled & True-Unlabeled Master Tables
# ============================================================

# ------------------------------------------------------------
# 1. Columns to retain in canonical master tables
# ------------------------------------------------------------

MASTER_REFERENCE_COLUMNS = [
    "id",
    "screen_name",
    "user_key",

    # raw text / reference information
    "name",
    "description",
    "clean_description",
    "created_at"
]

MASTER_FEATURE_COLUMNS = (
    FINAL_TABULAR_FEATURES
)

MASTER_COLUMNS = list(dict.fromkeys(
    MASTER_REFERENCE_COLUMNS
    + MASTER_FEATURE_COLUMNS
))


# ------------------------------------------------------------
# 2. Build clean label table from labeled 1K
# ------------------------------------------------------------

label_lookup = labeled_users[
    [
        "user_key",
        label_column
    ]
].copy()

label_lookup = label_lookup.rename(
    columns={
        label_column: "label_raw"
    }
)


# ------------------------------------------------------------
# 3. Create binary label
#
# human = 0
# bot   = 1
#
# Other classes remain NaN and will NOT be used in the
# main binary supervised task.
# ------------------------------------------------------------

binary_label_map = {
    "human(2)": 0,
    "bot(1)": 1
}

label_lookup["label_binary"] = (
    label_lookup["label_raw"]
    .map(binary_label_map)
)


# ------------------------------------------------------------
# 4. Build labeled master from canonical 19K user table
# ------------------------------------------------------------

labeled_master = (
    users_master[
        users_master["user_key"].isin(
            labeled_set
        )
    ][MASTER_COLUMNS]
    .merge(
        label_lookup,
        on="user_key",
        how="left",
        validate="one_to_one"
    )
    .copy()
)


# ------------------------------------------------------------
# 5. Build true-unlabeled master
# ------------------------------------------------------------

true_unlabeled_master = (
    users_master[
        ~users_master["user_key"].isin(
            labeled_set
        )
    ][MASTER_COLUMNS]
    .copy()
)

true_unlabeled_master["label_raw"] = pd.NA
true_unlabeled_master["label_binary"] = pd.NA


# ------------------------------------------------------------
# 6. Add modality availability masks
# ------------------------------------------------------------

def add_modality_masks(df):

    df = df.copy()

    # -------------------------
    # Profile modality
    # -------------------------

    profile_check_cols = [
        "followers_count",
        "friends_count",
        "user_age"
    ]

    df["has_profile"] = (
        df[profile_check_cols]
        .notna()
        .any(axis=1)
        .astype(int)
    )

    # -------------------------
    # Raw tweet modality
    # -------------------------

    df["has_raw_tweets"] = (
        df["user_key"]
        .isin(tweet_user_set)
        .astype(int)
    )

    # -------------------------
    # Description text
    # -------------------------

    clean_desc_present = (
        df["clean_description"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    raw_desc_present = (
        df["description"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    df["has_description"] = (
        clean_desc_present
        | raw_desc_present
    ).astype(int)

    # -------------------------
    # Behavioral / temporal modality
    # -------------------------

    df["has_behavior_temporal"] = (
        df[BEHAVIOR_TEMPORAL_FEATURES]
        .notna()
        .all(axis=1)
        .astype(int)
    )

    # -------------------------
    # Graph modality
    # -------------------------

    df["has_graph"] = (
        df["user_key"]
        .isin(graph_usable_user_set)
        .astype(int)
    )

    return df


labeled_master = add_modality_masks(
    labeled_master
)

true_unlabeled_master = add_modality_masks(
    true_unlabeled_master
)


# ------------------------------------------------------------
# 7. Build binary supervised dataset
# ------------------------------------------------------------

labeled_binary_master = (
    labeled_master[
        labeled_master["label_binary"]
        .notna()
    ]
    .copy()
)

labeled_binary_master[
    "label_binary"
] = labeled_binary_master[
    "label_binary"
].astype(int)


# ------------------------------------------------------------
# 8. Build complete multimodal binary subset
#
# Required:
#   binary label
#   profile
#   raw tweets
#   temporal/behavioral
#   graph
# ------------------------------------------------------------

complete_multimodal_binary = (
    labeled_binary_master[
        (
            labeled_binary_master["has_profile"] == 1
        )
        &
        (
            labeled_binary_master["has_raw_tweets"] == 1
        )
        &
        (
            labeled_binary_master["has_behavior_temporal"] == 1
        )
        &
        (
            labeled_binary_master["has_graph"] == 1
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 9. Report dataset sizes
# ------------------------------------------------------------

print("=" * 80)
print("CANONICAL MASTER DATASETS")
print("=" * 80)

print(
    f"Labeled master users:        "
    f"{len(labeled_master)}"
)

print(
    f"Binary Bot/Human users:      "
    f"{len(labeled_binary_master)}"
)

print(
    f"True unlabeled users:        "
    f"{len(true_unlabeled_master)}"
)

print(
    f"Complete multimodal binary:  "
    f"{len(complete_multimodal_binary)}"
)


# ------------------------------------------------------------
# 10. Binary label distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BINARY LABEL DISTRIBUTION")
print("=" * 80)

print(
    labeled_binary_master[
        "label_raw"
    ].value_counts()
)


# ------------------------------------------------------------
# 11. Modality coverage — labeled binary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODALITY COVERAGE — BINARY LABELED USERS")
print("=" * 80)

modality_columns = [
    "has_profile",
    "has_description",
    "has_raw_tweets",
    "has_behavior_temporal",
    "has_graph"
]

for col in modality_columns:

    count = int(
        labeled_binary_master[col].sum()
    )

    pct = (
        count
        / len(labeled_binary_master)
        * 100
    )

    print(
        f"{col:25s}: "
        f"{count:4d} "
        f"({pct:6.2f}%)"
    )


# ------------------------------------------------------------
# 12. Complete multimodal binary label distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPLETE MULTIMODAL BINARY LABEL DISTRIBUTION")
print("=" * 80)

print(
    complete_multimodal_binary[
        "label_raw"
    ].value_counts()
)


# ------------------------------------------------------------
# 13. Modality coverage — true unlabeled
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODALITY COVERAGE — TRUE UNLABELED USERS")
print("=" * 80)

for col in modality_columns:

    count = int(
        true_unlabeled_master[col].sum()
    )

    pct = (
        count
        / len(true_unlabeled_master)
        * 100
    )

    print(
        f"{col:25s}: "
        f"{count:5d} "
        f"({pct:6.2f}%)"
    )


# ------------------------------------------------------------
# 14. Critical leakage / overlap checks
# ------------------------------------------------------------

labeled_master_set = set(
    labeled_master["user_key"]
)

unlabeled_master_set = set(
    true_unlabeled_master["user_key"]
)

overlap_check = (
    labeled_master_set
    & unlabeled_master_set
)


assert len(labeled_master) == 1103

assert len(
    labeled_binary_master
) == 959

assert len(
    true_unlabeled_master
) == 18332

assert len(overlap_check) == 0

assert (
    labeled_master["user_key"]
    .is_unique
)

assert (
    true_unlabeled_master["user_key"]
    .is_unique
)


print("\n" + "=" * 80)
print("SANITY CHECKS")
print("=" * 80)

print(
    "Labeled ∩ True Unlabeled:",
    len(overlap_check)
)

print(
    "All dataset separation checks passed."
)

CANONICAL MASTER DATASETS
Labeled master users:        1103
Binary Bot/Human users:      959
True unlabeled users:        18332
Complete multimodal binary:  276

BINARY LABEL DISTRIBUTION
label_raw
human(2)    772
bot(1)      187
Name: count, dtype: int64

MODALITY COVERAGE — BINARY LABELED USERS
has_profile              :  959 (100.00%)
has_description          :  915 ( 95.41%)
has_raw_tweets           :  959 (100.00%)
has_behavior_temporal    :  959 (100.00%)
has_graph                :  276 ( 28.78%)

COMPLETE MULTIMODAL BINARY LABEL DISTRIBUTION
label_raw
human(2)    232
bot(1)       44
Name: count, dtype: int64

MODALITY COVERAGE — TRUE UNLABELED USERS
has_profile              : 18332 (100.00%)
has_description          : 11648 ( 63.54%)
has_raw_tweets           :     0 (  0.00%)
has_behavior_temporal    : 18332 (100.00%)
has_graph                :  3033 ( 16.54%)

SANITY CHECKS
Labeled ∩ True Unlabeled: 0
All dataset separation checks passed.


In [12]:
# ============================================================
# Cell 10 — Save Canonical Master Datasets
# ============================================================

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

LABELED_MASTER_PATH = (
    PROCESSED_DIR / "labeled_master.csv"
)

LABELED_BINARY_MASTER_PATH = (
    PROCESSED_DIR / "labeled_binary_master.csv"
)

TRUE_UNLABELED_MASTER_PATH = (
    PROCESSED_DIR / "true_unlabeled_master.csv"
)

COMPLETE_MULTIMODAL_BINARY_PATH = (
    PROCESSED_DIR / "complete_multimodal_binary.csv"
)

FEATURE_POLICY_PATH = (
    PROCESSED_DIR / "feature_policy.json"
)


# ------------------------------------------------------------
# Save master datasets
# ------------------------------------------------------------

labeled_master.to_csv(
    LABELED_MASTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

labeled_binary_master.to_csv(
    LABELED_BINARY_MASTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

true_unlabeled_master.to_csv(
    TRUE_UNLABELED_MASTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

complete_multimodal_binary.to_csv(
    COMPLETE_MULTIMODAL_BINARY_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Save feature policy / schema
# ------------------------------------------------------------

feature_policy = {

    "target": {
        "column": "label_binary",
        "human": 0,
        "bot": 1
    },

    "profile_numeric_features":
        PROFILE_NUMERIC_FEATURES,

    "profile_binary_features":
        PROFILE_BINARY_FEATURES,

    "profile_categorical_features":
        PROFILE_CATEGORICAL_FEATURES,

    "behavior_temporal_features":
        BEHAVIOR_TEMPORAL_FEATURES,

    "derived_features":
        DERIVED_FEATURES,

    "final_tabular_features":
        FINAL_TABULAR_FEATURES,

    "text_features":
        TEXT_FEATURES,

    "external_detector_columns_excluded":
        EXTERNAL_DETECTOR_COLUMNS,

    "annotation_columns_excluded":
        ANNOTATION_COLUMNS
}


with open(
    FEATURE_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_policy,
        f,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# Verify saved files
# ------------------------------------------------------------

print("=" * 80)
print("SAVED PHASE-2 MASTER FILES")
print("=" * 80)

saved_files = [
    LABELED_MASTER_PATH,
    LABELED_BINARY_MASTER_PATH,
    TRUE_UNLABELED_MASTER_PATH,
    COMPLETE_MULTIMODAL_BINARY_PATH,
    FEATURE_POLICY_PATH
]

for path in saved_files:

    print(
        f"{path.name:40s}",
        "OK" if path.exists() else "MISSING"
    )


# ------------------------------------------------------------
# File sizes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATASET SUMMARY")
print("=" * 80)

print(
    f"labeled_master:              "
    f"{labeled_master.shape}"
)

print(
    f"labeled_binary_master:       "
    f"{labeled_binary_master.shape}"
)

print(
    f"true_unlabeled_master:       "
    f"{true_unlabeled_master.shape}"
)

print(
    f"complete_multimodal_binary:  "
    f"{complete_multimodal_binary.shape}"
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

assert (
    pd.read_csv(
        LABELED_BINARY_MASTER_PATH,
        low_memory=False
    ).shape[0]
    == 959
)

assert (
    pd.read_csv(
        TRUE_UNLABELED_MASTER_PATH,
        low_memory=False
    ).shape[0]
    == 18332
)

assert (
    pd.read_csv(
        COMPLETE_MULTIMODAL_BINARY_PATH,
        low_memory=False
    ).shape[0]
    == 276
)


print("\nAll canonical datasets saved successfully.")

SAVED PHASE-2 MASTER FILES
labeled_master.csv                       OK
labeled_binary_master.csv                OK
true_unlabeled_master.csv                OK
complete_multimodal_binary.csv           OK
feature_policy.json                      OK

DATASET SUMMARY
labeled_master:              (1103, 61)
labeled_binary_master:       (959, 61)
true_unlabeled_master:       (18332, 61)
complete_multimodal_binary:  (276, 61)

All canonical datasets saved successfully.


In [13]:
# ============================================================
# Cell 11 — Data Type Cleaning & Validation
# ============================================================

# ------------------------------------------------------------
# 1. Define feature groups by expected data type
# ------------------------------------------------------------

NUMERIC_FEATURES = (
    PROFILE_NUMERIC_FEATURES
    + BEHAVIOR_TEMPORAL_FEATURES
    + DERIVED_FEATURES
)

BINARY_FEATURES = PROFILE_BINARY_FEATURES

CATEGORICAL_FEATURES = PROFILE_CATEGORICAL_FEATURES


print("=" * 80)
print("EXPECTED FEATURE TYPES")
print("=" * 80)

print("Numeric features:      ", len(NUMERIC_FEATURES))
print("Binary features:       ", len(BINARY_FEATURES))
print("Categorical features:  ", len(CATEGORICAL_FEATURES))

print(
    "Total:",
    len(NUMERIC_FEATURES)
    + len(BINARY_FEATURES)
    + len(CATEGORICAL_FEATURES)
)


# ------------------------------------------------------------
# 2. Robust binary conversion
# ------------------------------------------------------------

TRUE_VALUES = {
    "true",
    "1",
    "1.0",
    "yes",
    "y"
}

FALSE_VALUES = {
    "false",
    "0",
    "0.0",
    "no",
    "n"
}


def clean_binary_value(value):

    if pd.isna(value):
        return np.nan

    value_str = str(value).strip().lower()

    if value_str in TRUE_VALUES:
        return 1.0

    if value_str in FALSE_VALUES:
        return 0.0

    return np.nan


# ------------------------------------------------------------
# 3. Cleaning function
# ------------------------------------------------------------

def clean_master_dtypes(df, dataset_name="dataset"):

    df = df.copy()

    report = {
        "dataset": dataset_name,
        "numeric_coercion_to_nan": {},
        "binary_unknown_values": {},
        "infinite_values_before": 0,
        "infinite_values_after": 0
    }

    # --------------------------------------------------------
    # Numeric features
    # --------------------------------------------------------

    for col in NUMERIC_FEATURES:

        before_missing = df[col].isna().sum()

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        after_missing = df[col].isna().sum()

        newly_missing = (
            after_missing
            - before_missing
        )

        report[
            "numeric_coercion_to_nan"
        ][col] = int(newly_missing)

    # --------------------------------------------------------
    # Binary features
    # --------------------------------------------------------

    for col in BINARY_FEATURES:

        original_non_missing = (
            df[col]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
        )

        recognized = (
            original_non_missing.isin(
                TRUE_VALUES | FALSE_VALUES
            )
        )

        unknown_values = sorted(
            original_non_missing[
                ~recognized
            ]
            .unique()
            .tolist()
        )

        report[
            "binary_unknown_values"
        ][col] = unknown_values

        df[col] = (
            df[col]
            .apply(clean_binary_value)
            .astype("float64")
        )

    # --------------------------------------------------------
    # Categorical features
    # --------------------------------------------------------

    for col in CATEGORICAL_FEATURES:

        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        df[col] = df[col].replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "none": pd.NA,
                "<na>": pd.NA
            }
        )

    # --------------------------------------------------------
    # Detect infinity
    # --------------------------------------------------------

    numeric_array = (
        df[NUMERIC_FEATURES + BINARY_FEATURES]
        .to_numpy(dtype=float)
    )

    report[
        "infinite_values_before"
    ] = int(
        np.isinf(numeric_array).sum()
    )

    # Replace ±inf with NaN
    df[
        NUMERIC_FEATURES + BINARY_FEATURES
    ] = df[
        NUMERIC_FEATURES + BINARY_FEATURES
    ].replace(
        [np.inf, -np.inf],
        np.nan
    )

    numeric_array_after = (
        df[NUMERIC_FEATURES + BINARY_FEATURES]
        .to_numpy(dtype=float)
    )

    report[
        "infinite_values_after"
    ] = int(
        np.isinf(
            numeric_array_after
        ).sum()
    )

    # --------------------------------------------------------
    # Modality masks
    # --------------------------------------------------------

    modality_cols = [
        "has_profile",
        "has_description",
        "has_raw_tweets",
        "has_behavior_temporal",
        "has_graph"
    ]

    for col in modality_cols:
        df[col] = (
            pd.to_numeric(
                df[col],
                errors="coerce"
            )
            .astype("Int8")
        )

    return df, report


# ------------------------------------------------------------
# 4. Clean all master datasets consistently
# ------------------------------------------------------------

labeled_master_clean, report_labeled = (
    clean_master_dtypes(
        labeled_master,
        "labeled_master"
    )
)

labeled_binary_clean, report_binary = (
    clean_master_dtypes(
        labeled_binary_master,
        "labeled_binary_master"
    )
)

true_unlabeled_clean, report_unlabeled = (
    clean_master_dtypes(
        true_unlabeled_master,
        "true_unlabeled_master"
    )
)

complete_multimodal_binary_clean, report_complete = (
    clean_master_dtypes(
        complete_multimodal_binary,
        "complete_multimodal_binary"
    )
)


# ------------------------------------------------------------
# 5. Numeric conversion report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMERIC COERCION CHECK")
print("=" * 80)

for report in [
    report_labeled,
    report_unlabeled
]:

    problematic = {
        key: value
        for key, value
        in report[
            "numeric_coercion_to_nan"
        ].items()
        if value > 0
    }

    print(
        f"\n{report['dataset']}:"
    )

    if problematic:
        print(problematic)
    else:
        print(
            "No unexpected numeric values."
        )


# ------------------------------------------------------------
# 6. Binary value report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BINARY VALUE CHECK")
print("=" * 80)

for col in BINARY_FEATURES:

    unknown_labeled = (
        report_labeled[
            "binary_unknown_values"
        ][col]
    )

    unknown_unlabeled = (
        report_unlabeled[
            "binary_unknown_values"
        ][col]
    )

    print(
        f"{col:28s} | "
        f"labeled unknown: {unknown_labeled} | "
        f"unlabeled unknown: {unknown_unlabeled}"
    )


# ------------------------------------------------------------
# 7. Categorical value inspection
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CATEGORICAL FEATURE VALUES")
print("=" * 80)

for col in CATEGORICAL_FEATURES:

    print(f"\nFeature: {col}")

    print(
        pd.concat([
            labeled_binary_clean[col],
            true_unlabeled_clean[col]
        ])
        .value_counts(
            dropna=False
        )
    )


# ------------------------------------------------------------
# 8. Infinity check
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INFINITY CHECK")
print("=" * 80)

for report in [
    report_labeled,
    report_binary,
    report_unlabeled,
    report_complete
]:

    print(
        f"{report['dataset']:32s} | "
        f"before = "
        f"{report['infinite_values_before']:4d} | "
        f"after = "
        f"{report['infinite_values_after']:4d}"
    )


# ------------------------------------------------------------
# 9. Verify binary features now contain only 0, 1, NaN
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STANDARDIZED BINARY FEATURES")
print("=" * 80)

binary_validation = []

for col in BINARY_FEATURES:

    labeled_values = set(
        labeled_binary_clean[col]
        .dropna()
        .unique()
    )

    unlabeled_values = set(
        true_unlabeled_clean[col]
        .dropna()
        .unique()
    )

    valid_labeled = labeled_values.issubset(
        {0.0, 1.0}
    )

    valid_unlabeled = unlabeled_values.issubset(
        {0.0, 1.0}
    )

    binary_validation.append({
        "feature": col,
        "labeled_values":
            sorted(labeled_values),
        "unlabeled_values":
            sorted(unlabeled_values),
        "valid":
            valid_labeled
            and valid_unlabeled
    })


binary_validation = pd.DataFrame(
    binary_validation
)

display(binary_validation)


# ------------------------------------------------------------
# 10. Final integrity checks
# ------------------------------------------------------------

assert len(
    labeled_binary_clean
) == 959

assert len(
    true_unlabeled_clean
) == 18332

assert all(
    binary_validation["valid"]
)

assert set(
    FINAL_TABULAR_FEATURES
).issubset(
    labeled_binary_clean.columns
)

assert set(
    FINAL_TABULAR_FEATURES
).issubset(
    true_unlabeled_clean.columns
)


print("\n" + "=" * 80)
print("DATA TYPE CLEANING COMPLETE")
print("=" * 80)

print(
    "All 47 tabular features are present."
)

print(
    "No train/test information has been used."
)

print(
    "No imputer, scaler, or encoder has been fitted yet."
)

EXPECTED FEATURE TYPES
Numeric features:       37
Binary features:        9
Categorical features:   1
Total: 47

NUMERIC COERCION CHECK

labeled_master:
No unexpected numeric values.

true_unlabeled_master:
No unexpected numeric values.

BINARY VALUE CHECK
default_profile              | labeled unknown: [] | unlabeled unknown: []
default_profile_image        | labeled unknown: [] | unlabeled unknown: []
verified                     | labeled unknown: [] | unlabeled unknown: []
has_custom_timelines         | labeled unknown: [] | unlabeled unknown: []
is_translator                | labeled unknown: [] | unlabeled unknown: []
possibly_sensitive           | labeled unknown: [] | unlabeled unknown: []
want_retweets                | labeled unknown: [] | unlabeled unknown: []
hashtag_in_description       | labeled unknown: [] | unlabeled unknown: ['1.19363e+18']
numbers_in_description       | labeled unknown: [] | unlabeled unknown: ['1.19363e+18']

CATEGORICAL FEATURE VALUES

Feature: tran

,feature,labeled_values,unlabeled_values,valid
0,default_profile,"[0.0, 1.0]","[0.0, 1.0]",True
1,default_profile_image,"[0.0, 1.0]","[0.0, 1.0]",True
2,verified,[0.0],[0.0],True
3,has_custom_timelines,"[0.0, 1.0]","[0.0, 1.0]",True
4,is_translator,[0.0],[0.0],True
5,possibly_sensitive,"[0.0, 1.0]","[0.0, 1.0]",True
6,want_retweets,[0.0],[0.0],True
7,hashtag_in_description,"[0.0, 1.0]","[0.0, 1.0]",True
8,numbers_in_description,"[0.0, 1.0]","[0.0, 1.0]",True



DATA TYPE CLEANING COMPLETE
All 47 tabular features are present.
No train/test information has been used.
No imputer, scaler, or encoder has been fitted yet.


In [14]:
# ============================================================
# Cell 12 — Inspect Data Anomalies & Near-Constant Features
# ============================================================

# ------------------------------------------------------------
# 1. Locate invalid raw values in binary features
# ------------------------------------------------------------

print("=" * 80)
print("INVALID RAW BINARY VALUES")
print("=" * 80)

binary_anomaly_rows = []

for col in BINARY_FEATURES:

    raw_values = (
        users_master[col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )

    valid_mask = raw_values.isin(
        TRUE_VALUES | FALSE_VALUES
    )

    invalid_indices = raw_values[
        ~valid_mask
    ].index

    if len(invalid_indices) > 0:

        print(f"\nFeature: {col}")
        print(
            users_master.loc[
                invalid_indices,
                [
                    "id",
                    "screen_name",
                    "user_key",
                    col
                ]
            ]
        )

        for idx in invalid_indices:
            binary_anomaly_rows.append({
                "row_index": idx,
                "feature": col,
                "value": users_master.loc[idx, col],
                "user_key": users_master.loc[idx, "user_key"]
            })


binary_anomaly_report = pd.DataFrame(
    binary_anomaly_rows
)

print("\nTotal invalid binary cells:",
      len(binary_anomaly_report))


# ------------------------------------------------------------
# 2. If anomaly exists, inspect the full suspicious row
# ------------------------------------------------------------

if len(binary_anomaly_report) > 0:

    suspicious_indices = (
        binary_anomaly_report[
            "row_index"
        ]
        .unique()
    )

    print("\n" + "=" * 80)
    print("SUSPICIOUS RAW ROWS")
    print("=" * 80)

    for idx in suspicious_indices:

        print(f"\n--- Row index: {idx} ---")

        row = users_master.loc[idx]

        non_missing_row = row[
            row.notna()
        ]

        display(
            non_missing_row.to_frame(
                name="value"
            )
        )


# ------------------------------------------------------------
# 3. Check number of unique values for all 47 model features
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FEATURE CARDINALITY")
print("=" * 80)

cardinality_report = []

for col in FINAL_TABULAR_FEATURES:

    labeled_unique = (
        labeled_binary_clean[col]
        .nunique(dropna=True)
    )

    unlabeled_unique = (
        true_unlabeled_clean[col]
        .nunique(dropna=True)
    )

    combined_unique = (
        pd.concat([
            labeled_binary_clean[col],
            true_unlabeled_clean[col]
        ])
        .nunique(dropna=True)
    )

    cardinality_report.append({
        "feature": col,
        "labeled_unique": labeled_unique,
        "unlabeled_unique": unlabeled_unique,
        "combined_unique": combined_unique,
        "labeled_missing_pct":
            labeled_binary_clean[col]
            .isna()
            .mean() * 100,
        "unlabeled_missing_pct":
            true_unlabeled_clean[col]
            .isna()
            .mean() * 100
    })


cardinality_report = pd.DataFrame(
    cardinality_report
).sort_values(
    by=[
        "labeled_unique",
        "combined_unique"
    ]
)


display(
    cardinality_report
)


# ------------------------------------------------------------
# 4. Constant features in labeled binary dataset
# ------------------------------------------------------------

constant_labeled_features = (
    cardinality_report.loc[
        cardinality_report[
            "labeled_unique"
        ] <= 1,
        "feature"
    ]
    .tolist()
)


print("\n" + "=" * 80)
print("CONSTANT FEATURES — LABELED BINARY")
print("=" * 80)

print(constant_labeled_features)


# ------------------------------------------------------------
# 5. Extremely sparse categorical feature
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRANSLATOR_TYPE COVERAGE")
print("=" * 80)

for name, df in {
    "binary labeled": labeled_binary_clean,
    "true unlabeled": true_unlabeled_clean
}.items():

    present = df["translator_type"].notna().sum()
    total = len(df)

    print(
        f"{name:20s}: "
        f"{present:5d}/{total:5d} "
        f"({present / total * 100:.3f}%)"
    )

    print(
        df["translator_type"]
        .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# 6. Check whether suspicious row belongs to labeled/unlabeled
# ------------------------------------------------------------

if len(binary_anomaly_report) > 0:

    print("\n" + "=" * 80)
    print("ANOMALY DATASET MEMBERSHIP")
    print("=" * 80)

    anomaly_users = set(
        binary_anomaly_report[
            "user_key"
        ]
        .dropna()
    )

    print(
        "In labeled set:",
        len(
            anomaly_users
            & labeled_set
        )
    )

    print(
        "In true unlabeled set:",
        len(
            anomaly_users
            & true_unlabeled_set
        )
    )

INVALID RAW BINARY VALUES

Feature: hashtag_in_description
               id          screen_name             user_key  \
198  1.193630e+18  1193634420000993280  1193634420000993280   

     hashtag_in_description  
198            1.193630e+18  

Feature: numbers_in_description
               id          screen_name             user_key  \
198  1.193630e+18  1193634420000993280  1193634420000993280   

     numbers_in_description  
198            1.193630e+18  

Total invalid binary cells: 2

SUSPICIOUS RAW ROWS

--- Row index: 198 ---


,value
id,1193630000000000000.0
cluster_id,1193630000000000000.0
screen_name,1193634420000993280
name,1193634420000993280
clean_description,1193634420000993280
followers_count,1193630000000000000.0
friends_count,1193630000000000000.0
default_profile,1.0
default_profile_image,False
verified,0.0



FEATURE CARDINALITY


,feature,labeled_unique,unlabeled_unique,combined_unique,labeled_missing_pct,unlabeled_missing_pct
7,fast_followers_count,1,1,1,0.000000,0.005455
14,verified,1,1,1,0.000000,0.000000
16,is_translator,1,1,1,0.000000,0.005455
18,want_retweets,1,1,1,0.000000,0.005455
21,translator_type,1,1,1,98.331595,99.694523
12,default_profile,2,2,2,0.000000,0.000000
13,default_profile_image,2,2,2,0.000000,0.000000
15,has_custom_timelines,2,2,2,0.000000,0.005455
17,possibly_sensitive,2,2,2,0.000000,0.005455
19,hashtag_in_description,2,2,2,0.000000,0.005455



CONSTANT FEATURES — LABELED BINARY
['fast_followers_count', 'verified', 'is_translator', 'want_retweets', 'translator_type']

TRANSLATOR_TYPE COVERAGE
binary labeled      :    16/  959 (1.668%)
translator_type
<NA>       943
regular     16
Name: count, dtype: Int64
true unlabeled      :    56/18332 (0.305%)
translator_type
<NA>       18276
regular       56
Name: count, dtype: Int64

ANOMALY DATASET MEMBERSHIP
In labeled set: 0
In true unlabeled set: 1


In [15]:
# ============================================================
# Cell 13 — Remove Corrupt User & Finalize Model Feature Space
# ============================================================

# ------------------------------------------------------------
# 1. Explicitly record malformed unlabeled users
# ------------------------------------------------------------

CORRUPT_USER_KEYS = {
    "1193634420000993280"
}


print("=" * 80)
print("CORRUPT USER REMOVAL")
print("=" * 80)

print(
    "Corrupt users identified:",
    CORRUPT_USER_KEYS
)


# ------------------------------------------------------------
# 2. Remove corrupt users ONLY from true-unlabeled pool
# ------------------------------------------------------------

before_unlabeled = len(
    true_unlabeled_clean
)

true_unlabeled_clean = (
    true_unlabeled_clean[
        ~true_unlabeled_clean[
            "user_key"
        ].isin(CORRUPT_USER_KEYS)
    ]
    .copy()
)

after_unlabeled = len(
    true_unlabeled_clean
)


print(
    f"\nTrue unlabeled before: "
    f"{before_unlabeled}"
)

print(
    f"True unlabeled after:  "
    f"{after_unlabeled}"
)

print(
    f"Removed:               "
    f"{before_unlabeled - after_unlabeled}"
)


# ------------------------------------------------------------
# 3. Features removed from predictive model
# ------------------------------------------------------------

CONSTANT_OR_UNUSABLE_FEATURES = [
    "fast_followers_count",
    "verified",
    "is_translator",
    "want_retweets",
    "translator_type"
]


# ------------------------------------------------------------
# 4. Final predictive tabular feature space
# ------------------------------------------------------------

FINAL_MODEL_TABULAR_FEATURES = [
    feature
    for feature in FINAL_TABULAR_FEATURES
    if feature
    not in CONSTANT_OR_UNUSABLE_FEATURES
]


# Updated type groups
FINAL_NUMERIC_FEATURES = [
    feature
    for feature in NUMERIC_FEATURES
    if feature
    in FINAL_MODEL_TABULAR_FEATURES
]

FINAL_BINARY_FEATURES = [
    feature
    for feature in BINARY_FEATURES
    if feature
    in FINAL_MODEL_TABULAR_FEATURES
]

FINAL_CATEGORICAL_FEATURES = [
    feature
    for feature in CATEGORICAL_FEATURES
    if feature
    in FINAL_MODEL_TABULAR_FEATURES
]


# ------------------------------------------------------------
# 5. Report final feature counts
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL MODEL FEATURE SPACE")
print("=" * 80)

print(
    "Original tabular features:",
    len(FINAL_TABULAR_FEATURES)
)

print(
    "Removed constant/unusable:",
    len(CONSTANT_OR_UNUSABLE_FEATURES)
)

print(
    "Final tabular features:",
    len(FINAL_MODEL_TABULAR_FEATURES)
)

print(
    "\nFinal numeric:",
    len(FINAL_NUMERIC_FEATURES)
)

print(
    "Final binary:",
    len(FINAL_BINARY_FEATURES)
)

print(
    "Final categorical:",
    len(FINAL_CATEGORICAL_FEATURES)
)


print("\nRemoved features:")

for feature in CONSTANT_OR_UNUSABLE_FEATURES:
    print(" -", feature)


# ------------------------------------------------------------
# 6. Verify no remaining feature is constant in labeled data
# ------------------------------------------------------------

remaining_constant_features = []

for feature in FINAL_MODEL_TABULAR_FEATURES:

    n_unique = (
        labeled_binary_clean[
            feature
        ]
        .nunique(dropna=True)
    )

    if n_unique <= 1:
        remaining_constant_features.append(
            feature
        )


print("\n" + "=" * 80)
print("REMAINING CONSTANT FEATURE CHECK")
print("=" * 80)

print(
    remaining_constant_features
)

assert (
    len(
        remaining_constant_features
    )
    == 0
)


# ------------------------------------------------------------
# 7. Verify corrupt user is gone
# ------------------------------------------------------------

assert not any(
    true_unlabeled_clean[
        "user_key"
    ].isin(CORRUPT_USER_KEYS)
)


# ------------------------------------------------------------
# 8. Verify labeled dataset was NOT affected
# ------------------------------------------------------------

assert len(
    labeled_binary_clean
) == 959


# ------------------------------------------------------------
# 9. Final dataset sizes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL DATASET STATUS BEFORE SPLIT")
print("=" * 80)

print(
    "Binary labeled users:",
    len(labeled_binary_clean)
)

print(
    "Clean true-unlabeled users:",
    len(true_unlabeled_clean)
)

print(
    "Final model features:",
    len(FINAL_MODEL_TABULAR_FEATURES)
)


print(
    "\nFeature space finalized successfully."
)

CORRUPT USER REMOVAL
Corrupt users identified: {'1193634420000993280'}

True unlabeled before: 18332
True unlabeled after:  18331
Removed:               1

FINAL MODEL FEATURE SPACE
Original tabular features: 47
Removed constant/unusable: 5
Final tabular features: 42

Final numeric: 36
Final binary: 6
Final categorical: 0

Removed features:
 - fast_followers_count
 - verified
 - is_translator
 - want_retweets
 - translator_type

REMAINING CONSTANT FEATURE CHECK
[]

FINAL DATASET STATUS BEFORE SPLIT
Binary labeled users: 959
Clean true-unlabeled users: 18331
Final model features: 42

Feature space finalized successfully.


In [16]:
# ============================================================
# Cell 14 — Save Final QC-Cleaned Modeling Data
# ============================================================

# ------------------------------------------------------------
# 1. Columns that must remain in model-ready user tables
# ------------------------------------------------------------


MODEL_REFERENCE_COLUMNS = [
    "id",
    "screen_name",
    "user_key",
    "name",
    "description",
    "clean_description",
    "created_at"
]

MODEL_LABEL_COLUMNS = [
    "label_raw",
    "label_binary"
]

MODALITY_MASK_COLUMNS = [
    "has_profile",
    "has_description",
    "has_raw_tweets",
    "has_behavior_temporal",
    "has_graph"
]


MODEL_READY_COLUMNS = list(dict.fromkeys(
    MODEL_REFERENCE_COLUMNS
    + FINAL_MODEL_TABULAR_FEATURES
    + MODEL_LABEL_COLUMNS
    + MODALITY_MASK_COLUMNS
))


# ------------------------------------------------------------
# 2. Create trimmed model-ready datasets
# ------------------------------------------------------------

modeling_labeled_binary = (
    labeled_binary_clean[
        MODEL_READY_COLUMNS
    ]
    .copy()
)

modeling_true_unlabeled = (
    true_unlabeled_clean[
        MODEL_READY_COLUMNS
    ]
    .copy()
)

modeling_complete_multimodal = (
    complete_multimodal_binary_clean[
        MODEL_READY_COLUMNS
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. Output directory & paths
# ------------------------------------------------------------

PHASE2_DIR = PROCESSED_DIR / "phase2"
PHASE2_DIR.mkdir(parents=True, exist_ok=True)


MODELING_LABELED_PATH = (
    PHASE2_DIR /
    "modeling_labeled_binary.csv"
)

MODELING_UNLABELED_PATH = (
    PHASE2_DIR /
    "modeling_true_unlabeled.csv"
)

MODELING_COMPLETE_PATH = (
    PHASE2_DIR /
    "modeling_complete_multimodal_binary.csv"
)

MODEL_FEATURE_POLICY_PATH = (
    PHASE2_DIR /
    "model_feature_policy.json"
)

DATA_QUALITY_EXCLUSIONS_PATH = (
    PHASE2_DIR /
    "data_quality_exclusions.json"
)

print("Phase 2 output directory:")
print(PHASE2_DIR)


# ------------------------------------------------------------
# 4. Save final QC-cleaned datasets
# ------------------------------------------------------------

modeling_labeled_binary.to_csv(
    MODELING_LABELED_PATH,
    index=False,
    encoding="utf-8-sig"
)

modeling_true_unlabeled.to_csv(
    MODELING_UNLABELED_PATH,
    index=False,
    encoding="utf-8-sig"
)

modeling_complete_multimodal.to_csv(
    MODELING_COMPLETE_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 5. Save FINAL model feature policy
# ------------------------------------------------------------

model_feature_policy = {

    "target": {
        "column": "label_binary",
        "human": 0,
        "bot": 1
    },

    "final_model_tabular_features":
        FINAL_MODEL_TABULAR_FEATURES,

    "numeric_features":
        FINAL_NUMERIC_FEATURES,

    "binary_features":
        FINAL_BINARY_FEATURES,

    "categorical_features":
        FINAL_CATEGORICAL_FEATURES,

    "text_fields": [
        "description",
        "clean_description"
    ],

    "modality_masks":
        MODALITY_MASK_COLUMNS,

    "removed_constant_or_unusable_features":
        CONSTANT_OR_UNUSABLE_FEATURES,

    "external_detector_features_excluded":
        EXTERNAL_DETECTOR_COLUMNS,

    "annotation_columns_excluded":
        ANNOTATION_COLUMNS,

    "n_final_tabular_features":
        len(FINAL_MODEL_TABULAR_FEATURES)
}


with open(
    MODEL_FEATURE_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        model_feature_policy,
        f,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# 6. Save data-quality exclusions
# ------------------------------------------------------------

data_quality_exclusions = {
    "corrupt_user_keys": sorted(
        CORRUPT_USER_KEYS
    ),

    "reason": (
        "Malformed row containing repeated Twitter/X ID "
        "values across numerous unrelated profile and "
        "behavioral features."
    ),

    "raw_true_unlabeled_count": 18332,
    "clean_true_unlabeled_count": 18331
}


with open(
    DATA_QUALITY_EXCLUSIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data_quality_exclusions,
        f,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# 7. Verification report
# ------------------------------------------------------------

print("=" * 80)
print("FINAL MODEL-READY DATASETS")
print("=" * 80)

print(
    "Labeled binary:",
    modeling_labeled_binary.shape
)

print(
    "True unlabeled:",
    modeling_true_unlabeled.shape
)

print(
    "Complete multimodal binary:",
    modeling_complete_multimodal.shape
)

print(
    "\nFinal tabular features:",
    len(FINAL_MODEL_TABULAR_FEATURES)
)


print("\n" + "=" * 80)
print("FINAL FEATURE TYPES")
print("=" * 80)

print(
    "Numeric:",
    len(FINAL_NUMERIC_FEATURES)
)

print(
    "Binary:",
    len(FINAL_BINARY_FEATURES)
)

print(
    "Categorical:",
    len(FINAL_CATEGORICAL_FEATURES)
)


# ------------------------------------------------------------
# 8. Critical integrity checks
# ------------------------------------------------------------

assert len(modeling_labeled_binary) == 959

assert len(modeling_true_unlabeled) == 18331

assert len(modeling_complete_multimodal) == 276

assert len(FINAL_MODEL_TABULAR_FEATURES) == 42

assert not any(
    modeling_true_unlabeled[
        "user_key"
    ].isin(CORRUPT_USER_KEYS)
)

assert set(
    CONSTANT_OR_UNUSABLE_FEATURES
).isdisjoint(
    set(FINAL_MODEL_TABULAR_FEATURES)
)

assert set(
    EXTERNAL_DETECTOR_COLUMNS
).isdisjoint(
    set(modeling_labeled_binary.columns)
)


print("\n" + "=" * 80)
print("SAVED FILES")
print("=" * 80)

for path in [
    MODELING_LABELED_PATH,
    MODELING_UNLABELED_PATH,
    MODELING_COMPLETE_PATH,
    MODEL_FEATURE_POLICY_PATH,
    DATA_QUALITY_EXCLUSIONS_PATH
]:

    print(
        f"{path.name:45s}",
        "OK" if path.exists() else "MISSING"
    )


print(
    "\nFinal QC-cleaned modeling datasets saved successfully."
)

Phase 2 output directory:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\processed_data\phase2
FINAL MODEL-READY DATASETS
Labeled binary: (959, 56)
True unlabeled: (18331, 56)
Complete multimodal binary: (276, 56)

Final tabular features: 42

FINAL FEATURE TYPES
Numeric: 36
Binary: 6
Categorical: 0

SAVED FILES
modeling_labeled_binary.csv                   OK
modeling_true_unlabeled.csv                   OK
modeling_complete_multimodal_binary.csv       OK
model_feature_policy.json                     OK
data_quality_exclusions.json                  OK

Final QC-cleaned modeling datasets saved successfully.


In [18]:
# ============================================================
# Cell 15 — Inspect Final Operational Datasets
# ============================================================

from pathlib import Path
import pandas as pd
import json

# ------------------------------------------------------------
# 1. Final data directory
# ------------------------------------------------------------

FINAL_DATA_DIR = PROJECT_ROOT / "final_data"

USERS_DIR = FINAL_DATA_DIR / "users"
TWEETS_DIR = FINAL_DATA_DIR / "tweets"
GRAPH_DIR = FINAL_DATA_DIR / "graph"
CONFIG_DIR = FINAL_DATA_DIR / "config"


# ------------------------------------------------------------
# 2. Define final operational files
# ------------------------------------------------------------

FINAL_FILES = {

    "Labeled Binary Users":
        USERS_DIR / "modeling_labeled_binary.csv",

    "True Unlabeled Users":
        USERS_DIR / "modeling_true_unlabeled.csv",

    "Complete Multimodal Binary":
        USERS_DIR / "modeling_complete_multimodal_binary.csv",

    "Tweets":
        TWEETS_DIR / "tweets_meta_data.csv",

    "Graph Edges":
        GRAPH_DIR / "graph_edges.csv",

    "Graph User Statistics":
        GRAPH_DIR / "graph_user_statistics.csv"
}


# ------------------------------------------------------------
# 3. Inspect CSV datasets
# ------------------------------------------------------------

print("=" * 100)
print("FINAL OPERATIONAL DATASET INSPECTION")
print("=" * 100)

final_datasets = {}

for name, path in FINAL_FILES.items():

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    print("Path:")
    print(path)

    print("\nExists:", path.exists())

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(
        path,
        low_memory=False
    )

    final_datasets[name] = df

    print("\nShape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")

    display(
        df.head(5)
    )


# ------------------------------------------------------------
# 4. Inspect configuration JSON files
# ------------------------------------------------------------

CONFIG_FILES = {

    "Model Feature Policy":
        CONFIG_DIR / "model_feature_policy.json",

    "Data Quality Exclusions":
        CONFIG_DIR / "data_quality_exclusions.json"
}


print("\n" + "=" * 100)
print("FINAL CONFIGURATION FILES")
print("=" * 100)

for name, path in CONFIG_FILES.items():

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)

    print("Path:")
    print(path)

    print("\nExists:", path.exists())

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        content = json.load(f)

    print("\nContent:")

    print(
        json.dumps(
            content,
            ensure_ascii=False,
            indent=2
        )
    )

FINAL OPERATIONAL DATASET INSPECTION

Labeled Binary Users
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\users\modeling_labeled_binary.csv

Exists: True

Shape:
(959, 56)

Columns:
['id', 'screen_name', 'user_key', 'name', 'description', 'clean_description', 'created_at', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'status_count', 'normal_followers_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap

,id,screen_name,user_key,name,description,clean_description,created_at,followers_count,friends_count,favourites_count,...,num_digits_in_name,num_digits_in_username,url_in_description,label_raw,label_binary,has_profile,has_description,has_raw_tweets,has_behavior_temporal,has_graph
0,1.707640e+18,KathyMobarez,kathymobarez,Kathy Mobarez,NaN,NaN,2023-09-29T06:22:15+00:00,3586.0,1672.0,310972.0,...,0,0,0,bot(1),1,1,0,1,1,0
1,1.530060e+18,rahe_AZADI_5,rahe_azadi_5,KRATOS Spartan Rage👑,NaN,NaN,2022-05-27T05:48:47+00:00,NaN,NaN,288142.0,...,0,1,0,bot(1),1,1,0,1,1,0
2,1.391600e+18,dadkhahim,dadkhahim,دادخواهی‌م,#کشتار۶۷\n#تیر۷۸\n#خرداد۸۸\n#دی۹۶\n#آبان۹۸\n#ه...,NaN,2021-05-10T03:52:12+00:00,NaN,NaN,8895.0,...,0,0,0,bot(1),1,1,1,1,1,0
3,1.568300e+18,HiwaTubeAI,hiwatubeai,Hiwa,شاه، میهن، پرچم,The official handle of the Republic Media Netw...,2022-09-09T18:23:28+00:00,2841497.0,8.0,288898.0,...,0,0,0,bot(1),1,1,1,1,1,0
4,1.464650e+18,kazeroonkings,kazeroonkings,@kazeroonkings,ملت ایران ۹۹ دشمن و یک دوست بنام ملکه شهبانوفر...,ملت ایران ۹۹ دشمن و یک دوست بنام ملکه شهبانوفر...,2021-11-27T17:24:09+00:00,6830.0,5175.0,217554.0,...,0,0,0,bot(1),1,1,1,1,1,0



True Unlabeled Users
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\users\modeling_true_unlabeled.csv

Exists: True

Shape:
(18331, 56)

Columns:
['id', 'screen_name', 'user_key', 'name', 'description', 'clean_description', 'created_at', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'status_count', 'normal_followers_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap', 'unique_mention_rate_per_tweet',

,id,screen_name,user_key,name,description,clean_description,created_at,followers_count,friends_count,favourites_count,...,num_digits_in_name,num_digits_in_username,url_in_description,label_raw,label_binary,has_profile,has_description,has_raw_tweets,has_behavior_temporal,has_graph
0,6.178539e+08,PatriotPointman,patriotpointman,Brett Murphy,Writer & Poet | Videos & Live Streams | Americ...,Writer & Poet | Videos & Live Streams | Americ...,2012-06-25T04:38:20+00:00,159406.0,39400.0,42127.0,...,0,0,0,NaN,NaN,1,1,0,1,0
1,1.580000e+18,IranZiba1401,iranziba1401,Free Iran,"💚🦁🤍🌞❤️\n\n@PahlaviReza, https://t.co/dlBsD3cnj...","@PahlaviReza, \n @PahlaviComms, \n @ShahbanouF...",2022-10-12T00:53:19+00:00,2020.0,1938.0,49690.0,...,0,4,1,NaN,NaN,1,1,0,1,0
2,1.716080e+18,MIIran20194,miiran20194,دخترایران از نسل رستم,من دخترایران \nاز نسل رستم\nمیجنگم برای ایران ...,من دخترایران \n از نسل رستم\n میجنگم برای ایرا...,2023-10-22T13:33:33+00:00,917.0,558.0,68844.0,...,0,5,0,NaN,NaN,1,1,0,1,0
3,1.684610e+18,Artemis540721,artemis540721,👑Artemis👑,تا ابد و یک روز جاوید شاه،پاینده ایران,تا ابد و یک روز جاوید شاه،پاینده ایران,2023-07-27T17:19:20+00:00,1052.0,266.0,244259.0,...,0,6,0,NaN,NaN,1,1,0,1,0
4,1.800450e+18,tifraghe_n,tifraghe_n,Tifraghe,parody and political satire,parody and political satire,2024-06-11T08:55:49+00:00,2838.0,1826.0,120843.0,...,0,0,0,NaN,NaN,1,1,0,1,0



Complete Multimodal Binary
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\users\modeling_complete_multimodal_binary.csv

Exists: True

Shape:
(276, 56)

Columns:
['id', 'screen_name', 'user_key', 'name', 'description', 'clean_description', 'created_at', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'status_count', 'normal_followers_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap', 'unique_mention_

...


Graph Edges
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\graph\graph_edges.csv

Exists: True

Shape:
(796485, 6)

Columns:
['source', 'target', 'relation', 'source_in_dataset', 'target_in_dataset', 'internal_edge']

First 5 rows:


,source,target,relation,source_in_dataset,target_in_dataset,internal_edge
0,chaiiee,SajjadSade48567,follows,True,True,True
1,dr_lizzii,SajjadSade48567,follows,False,True,False
2,nathaniel2026,SajjadSade48567,follows,False,True,False
3,julian270zi,SajjadSade48567,follows,False,True,False
4,kongoshadha,SajjadSade48567,follows,True,True,True



Graph User Statistics
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\graph\graph_user_statistics.csv

Exists: True

Shape:
(3443, 8)

Columns:
['id', 'screen_name', 'followers', 'following', 'followers_list', 'following_list', 'collected_followers_count', 'collected_following_count']

First 5 rows:


,id,screen_name,followers,following,followers_list,following_list,collected_followers_count,collected_following_count
0,1.780000e+18,SajjadSade48567,"chaiiee, dr_lizzii, nathaniel2026, julian270zi...","grok, USABehFarsi, Psiphon_Fa, PersianDJT, Pho...","['chaiiee', 'dr_lizzii', 'nathaniel2026', 'jul...","['grok', 'USABehFarsi', 'Psiphon_Fa', 'Persian...",7,42
1,2.511595e+08,mojtaba2a,"aryan78093797, BanihashemiNav1, Riseagain979, ...","behrouzina, twiterrism819, monikaa2500, aryame...","['aryan78093797', 'BanihashemiNav1', 'Riseagai...","['behrouzina', 'twiterrism819', 'monikaa2500',...",193,160
2,1.790000e+18,niya64_,"Mentttaallll, TalbB909091, santinokarimi, Java...","chizbiin, Alireza774365, zahraaa1988, nooshiin...","['Mentttaallll', 'TalbB909091', 'santinokarimi...","['chizbiin', 'Alireza774365', 'zahraaa1988', '...",206,189
3,1.870000e+18,arash_the3rd,"Avayiran, darkpixele, sjavidnia, AnahitaSarab,...","pirozzzzzzzzz, Earendilist, salargholamiii, si...","['Avayiran', 'darkpixele', 'sjavidnia', 'Anahi...","['pirozzzzzzzzz', 'Earendilist', 'salargholami...",179,162
4,1.520000e+18,kaveAhanga2022,"leovirgo_17_, yutaabm, ShahdadmmMajid, Aliasad...","FanpageYaspah, leovirgo_17_, seyedhadikasaei, ...","['leovirgo_17_', 'yutaabm', 'ShahdadmmMajid', ...","['FanpageYaspah', 'leovirgo_17_', 'seyedhadika...",166,111



FINAL CONFIGURATION FILES

----------------------------------------------------------------------------------------------------
Model Feature Policy
----------------------------------------------------------------------------------------------------
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\config\model_feature_policy.json

Exists: True

Content:
{
  "target": {
    "column": "label_binary",
    "human": 0,
    "bot": 1
  },
  "final_model_tabular_features": [
    "followers_count",
    "friends_count",
    "favourites_count",
    "listed_count",
    "media_count",
    "statuses_count",
    "status_count",
    "normal_followers_count",
    "user_age",
    "follower_growth_rate",
    "friends_growth_rate",
    "default_profile",
    "default_profile_image",
    "has_custom_timelines",
    "possibly_sensitive",
    "hashtag_in_description",
    "numbers_in_description",
    "no_type_tweet",
    "no_type_retweet_with_comment",
    "no_typ